# Beyond GDP: Human Development and Capability Analysis

## Notebook 2 — Database and SQL Analysis

This notebook loads the cleaned master dataset, builds a small SQLite database
with four tables, and runs SQL queries to study the data.

Tables:
- hdi_data — HDI and its components
- happiness_data — happiness score and related factors
- economic_data — GDP, health spending, unemployment, inflation, population
- corruption_data — corruption perceptions index score

All four tables share the ISO3 country code, which is used to join them.

## 1. Load data and connect to the database

We load the cleaned master dataset and open a connection to a SQLite database
file. SQLite comes built into Python, so nothing needs to be installed. The
database file is created in the project root.

In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

master_path = Path(r"D:\beyond-gdp-capability-analysis\data\processed\master_dataset.csv")
master = pd.read_csv(master_path)

print("Shape:", master.shape)
print("Columns:", master.columns.tolist())

db_path = Path(r"D:\beyond-gdp-capability-analysis\capability_analysis.db")
conn = sqlite3.connect(db_path)
print("Connected to:", db_path.name)

Shape: (130, 18)
Columns: ['country', 'code', 'hdi', 'life_expectancy', 'expected_schooling', 'mean_schooling', 'gni_per_capita', 'happiness_score', 'social_support_contrib', 'freedom_contrib', 'generosity_contrib', 'corruption_contrib', 'gdp_per_capita', 'health_expenditure', 'unemployment', 'inflation', 'population', 'cpi_score']
Connected to: capability_analysis.db


## 2. Create the four tables

We split the master dataset into four topic tables and write each one into the
database. Every table keeps country and code so the tables can be joined later
on the code. We use if_exists="replace" so re-running this cell rebuilds the
tables cleanly instead of adding duplicate rows.

In [2]:
hdi_data = master[["code", "country", "hdi", "life_expectancy",
                   "expected_schooling", "mean_schooling", "gni_per_capita"]]

happiness_data = master[["code", "country", "happiness_score",
                         "social_support_contrib", "freedom_contrib",
                         "generosity_contrib", "corruption_contrib"]]

economic_data = master[["code", "country", "gdp_per_capita", "health_expenditure",
                        "unemployment", "inflation", "population"]]

corruption_data = master[["code", "country", "cpi_score"]]

hdi_data.to_sql("hdi_data", conn, if_exists="replace", index=False)
happiness_data.to_sql("happiness_data", conn, if_exists="replace", index=False)
economic_data.to_sql("economic_data", conn, if_exists="replace", index=False)
corruption_data.to_sql("corruption_data", conn, if_exists="replace", index=False)

print("Tables created.")

Tables created.


## 3. Check the tables

We ask the database to list its tables, then read a few rows from one table to
confirm the data was written correctly. From here on we write SQL and run it with
pandas, which returns the result as a table.

In [3]:
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(tables)

print()

pd.read_sql("SELECT * FROM hdi_data LIMIT 5;", conn)

              name
0         hdi_data
1   happiness_data
2    economic_data
3  corruption_data



,code,country,hdi,life_expectancy,expected_schooling,mean_schooling,gni_per_capita
0,ISL,Iceland,0.972,82.691,18.850590,13.908926,69116.93736
1,NOR,Norway,0.970,83.308,18.792850,13.117962,112710.02110
2,CHE,Switzerland,0.970,83.954,16.667530,13.949121,81948.90177
3,DNK,Denmark,0.962,81.933,18.704010,13.027321,76007.85669
4,DEU,Germany,0.959,81.378,17.309219,14.296372,64053.22124


## 4. SQL analysis

### Q1. Top 10 countries by HDI

We sort the HDI table from highest to lowest HDI and take the top 10.

In [4]:
query = """
SELECT country, hdi, life_expectancy, gni_per_capita
FROM hdi_data
ORDER BY hdi DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,country,hdi,life_expectancy,gni_per_capita
0,Iceland,0.972,82.691,69116.93736
1,Norway,0.970,83.308,112710.02110
2,Switzerland,0.970,83.954,81948.90177
3,Denmark,0.962,81.933,76007.85669
4,Germany,0.959,81.378,64053.22124
5,Sweden,0.959,83.262,66102.08595
6,Australia,0.958,83.923,58276.87643
7,Netherlands,0.955,82.158,68344.30826
8,Belgium,0.951,82.115,63582.47535
9,Ireland,0.949,82.412,90884.63459


**Conclusion (Q1):**

We sorted countries by HDI (development level) from highest to lowest and took the
top 10. Iceland is first with an HDI of 0.972, followed by Norway, Switzerland,
Denmark and Germany. All ten are rich, mostly European countries.

In plain words: the most developed countries in our data are wealthy nations with
long life expectancy and high schooling. No surprise here, but it gives us a clear
top end to compare against later.

### Q2. Top 10 countries by happiness score

We sort the happiness table by happiness score, highest first, and take the top 10.

In [13]:
query = """
SELECT country, happiness_score, social_support_contrib, freedom_contrib
FROM happiness_data
ORDER BY happiness_score DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,country,happiness_score,social_support_contrib,freedom_contrib
0,Finland,7.741,1.572,0.859
1,Denmark,7.583,1.520,0.823
2,Iceland,7.525,1.617,0.819
3,Sweden,7.344,1.501,0.838
4,Israel,7.341,1.513,0.641
5,Netherlands,7.319,1.462,0.725
6,Norway,7.302,1.517,0.835
7,Luxembourg,7.122,1.355,0.801
8,Switzerland,7.060,1.425,0.759
9,Australia,7.057,1.461,0.756


**Conclusion (Q2):**

Here we sorted by happiness score instead of HDI. Finland is the happiest (7.741),
followed by Denmark, Iceland and Sweden, the Nordic countries.

Notice that this list looks a lot like the HDI list, but the order is different.
For example Iceland is first on HDI but third on happiness, and Finland is first
on happiness but not at the top of HDI. So development and happiness are related,
but they are not the same thing. This small difference is exactly what the project
wants to explore.

### Q3. High GDP but relatively lower HDI

The core question of this project is whether income alone explains development.
Here we join the HDI and economic tables on the country code, then look for
countries with high GDP per capita but a HDI that is not as high. We rank both
GDP and HDI, and look at the gap between the two ranks.

In [14]:
query = """
SELECT
    h.country,
    e.gdp_per_capita,
    h.hdi
FROM hdi_data AS h
JOIN economic_data AS e
    ON h.code = e.code
ORDER BY e.gdp_per_capita DESC
LIMIT 15;
"""

pd.read_sql(query, conn)

,country,gdp_per_capita,hdi
0,Luxembourg,133230.619179,0.922
1,Ireland,106818.917131,0.949
2,Switzerland,100623.549627,0.970
3,Norway,87497.217965,0.970
4,Singapore,85412.230345,0.946
5,Iceland,82138.789297,0.972
6,United States,81032.262118,0.938
7,Denmark,68043.546697,0.962
8,Australia,65058.377315,0.958
9,Netherlands,63515.603078,0.955


**Conclusion (Q3):**

We joined the HDI table and the economic table so each country shows its GDP per
capita and its HDI side by side, then sorted by GDP.

Luxembourg has the highest GDP (about 133,000) but its HDI is 0.922, while Iceland
has roughly half the GDP (about 82,000) but a higher HDI of 0.972. So more money
does not automatically mean more development.

This is the first hint of the project's main idea: income and development do not
always line up. The next question measures this gap properly.

### Q4. Gap between GDP rank and HDI rank

We rank every country by GDP per capita and separately by HDI, both from highest
to lowest. Then we compute (HDI rank − GDP rank). A large positive value means the
country ranks much worse on HDI than on income, that is, high GDP but comparatively
low HDI. We show the countries with the biggest such gap.

In [7]:
query = """
WITH ranked AS (
    SELECT
        h.country,
        e.gdp_per_capita,
        h.hdi,
        RANK() OVER (ORDER BY e.gdp_per_capita DESC) AS gdp_rank,
        RANK() OVER (ORDER BY h.hdi DESC) AS hdi_rank
    FROM hdi_data AS h
    JOIN economic_data AS e
        ON h.code = e.code
)
SELECT
    country,
    gdp_per_capita,
    hdi,
    gdp_rank,
    hdi_rank,
    hdi_rank - gdp_rank AS rank_gap
FROM ranked
ORDER BY rank_gap DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,country,gdp_per_capita,hdi,gdp_rank,hdi_rank,rank_gap
0,Luxembourg,133230.619179,0.922,1,23,22
1,Jamaica,7542.400843,0.720,67,87,20
2,Botswana,7826.353765,0.731,65,84,19
3,Guatemala,5758.327608,0.662,78,97,19
4,Kuwait,34075.849013,0.852,27,45,18
5,Gabon,7802.824333,0.733,66,82,16
6,Iraq,5965.318351,0.695,76,92,16
7,Mexico,13860.959781,0.789,49,64,15
8,El Salvador,5365.444914,0.678,80,95,15
9,Dominican Republic,10630.431744,0.776,57,70,13


**Conclusion (Q4):**

Instead of judging by eye, we gave each country a rank for GDP and a separate rank
for HDI, then looked at (HDI rank minus GDP rank). A big positive number means the
country ranks much worse on development than on income, in other words, high money
but comparatively low development.

The biggest gaps include Luxembourg, Kuwait, Botswana, Gabon and Iraq. These are
countries where income sits higher than their human development.

One honest point: this is a relative comparison. Luxembourg still has a very high
HDI of 0.922, so it is not "undeveloped", it just ranks lower on HDI than its huge
income would suggest. The gap shows priorities, not failure.

### Q5. Lower GDP but relatively higher HDI

The reverse of the previous question. A large negative (HDI rank − GDP rank) means
a country ranks much better on HDI than its income would suggest, that is, it
converts limited income into strong human development. These cases are central to
the capability argument.

In [15]:
query = """
WITH ranked AS (
    SELECT
        h.country,
        e.gdp_per_capita,
        h.hdi,
        RANK() OVER (ORDER BY e.gdp_per_capita DESC) AS gdp_rank,
        RANK() OVER (ORDER BY h.hdi DESC) AS hdi_rank
    FROM hdi_data AS h
    JOIN economic_data AS e
        ON h.code = e.code
)
SELECT
    country,
    gdp_per_capita,
    hdi,
    gdp_rank,
    hdi_rank,
    hdi_rank - gdp_rank AS rank_gap
FROM ranked
ORDER BY rank_gap ASC
LIMIT 10;
"""

pd.read_sql(query, conn)

,country,gdp_per_capita,hdi,gdp_rank,hdi_rank,rank_gap
0,Iran (Islamic Republic of),5049.299316,0.799,81,60,-21
1,Sri Lanka,3798.890166,0.776,90,70,-20
2,Egypt,3456.789685,0.754,92,75,-17
3,Kyrgyzstan,2138.222102,0.720,103,87,-16
4,Georgia,8283.669607,0.844,62,48,-14
5,Lebanon,3477.724898,0.752,91,77,-14
6,Uzbekistan,2878.968793,0.740,94,81,-13
7,Montenegro,12259.877503,0.862,54,42,-12
8,Viet Nam,4323.350320,0.766,84,72,-12
9,Slovenia,32660.475358,0.931,30,19,-11


**Conclusion (Q5):**

This is the reverse of Q4. A big negative gap means a country ranks much better on
development than its income would suggest, that is, it turns limited money into
strong human development.

The clearest cases are Iran, Sri Lanka, Egypt, Vietnam and Uzbekistan. These
countries have relatively low GDP but better HDI ranks, often because they invested
in health and education over many years.

Q4 and Q5 together make the project's main point: money and development do not move
hand in hand. Some rich countries underperform on development, and some poorer
countries punch above their income.

### Q6. Group countries by HDI level, then compare averages

There is no region column in the data, so we first create a development category
from the HDI value using the standard UNDP cut-offs (CASE WHEN). Then we group by
that category and compare the average GDP, happiness and corruption score across
the four groups. This shows how the dimensions move together at the group level.

In [9]:
query = """
SELECT
    CASE
        WHEN h.hdi >= 0.800 THEN '1 Very High'
        WHEN h.hdi >= 0.700 THEN '2 High'
        WHEN h.hdi >= 0.550 THEN '3 Medium'
        ELSE '4 Low'
    END AS hdi_group,
    COUNT(*) AS n_countries,
    ROUND(AVG(e.gdp_per_capita), 0) AS avg_gdp,
    ROUND(AVG(hp.happiness_score), 2) AS avg_happiness,
    ROUND(AVG(c.cpi_score), 1) AS avg_cpi
FROM hdi_data AS h
JOIN economic_data AS e   ON h.code = e.code
JOIN happiness_data AS hp ON h.code = hp.code
JOIN corruption_data AS c ON h.code = c.code
GROUP BY hdi_group
ORDER BY hdi_group;
"""

pd.read_sql(query, conn)

,hdi_group,n_countries,avg_gdp,avg_happiness,avg_cpi
0,1 Very High,59,38649.0,6.46,58.9
1,2 High,32,6132.0,5.37,33.3
2,3 Medium,23,2519.0,4.66,31.2
3,4 Low,16,949.0,4.20,31.2


**Conclusion (Q6):**

We split countries into four HDI groups and compared the average GDP, happiness and
corruption score of each group.

The averages:
- Very High: GDP 38,649, happiness 6.46, CPI 58.9
- High: GDP 6,132, happiness 5.37, CPI 33.3
- Medium: GDP 2,519, happiness 4.66, CPI 31.2
- Low: GDP 949, happiness 4.20, CPI 31.2

Two things stand out. First, the GDP gap between groups is huge, the top group
earns about 40 times the bottom group. Second, happiness and corruption do not fall
nearly as sharply. Happiness only drops from 6.46 to 4.20, and the Medium and Low
groups have almost the same corruption score (31.2).

In plain words: income separates countries strongly, but happiness and governance
do not follow income in the same steep way. Money is one part of the story, not the
whole story.

### Q7. Keep only the larger HDI groups (HAVING)

WHERE filters individual rows before grouping. HAVING filters groups after they
are formed. Here we group by HDI level and then keep only groups that contain more
than 20 countries.

In [10]:
query = """
SELECT
    CASE
        WHEN hdi >= 0.800 THEN '1 Very High'
        WHEN hdi >= 0.700 THEN '2 High'
        WHEN hdi >= 0.550 THEN '3 Medium'
        ELSE '4 Low'
    END AS hdi_group,
    COUNT(*) AS n_countries
FROM hdi_data
GROUP BY hdi_group
HAVING COUNT(*) > 20
ORDER BY hdi_group;
"""

pd.read_sql(query, conn)

,hdi_group,n_countries
0,1 Very High,59
1,2 High,32
2,3 Medium,23


In [11]:
check = pd.read_sql("""
SELECT
    CASE
        WHEN hdi >= 0.800 THEN '1 Very High'
        WHEN hdi >= 0.700 THEN '2 High'
        WHEN hdi >= 0.550 THEN '3 Medium'
        ELSE '4 Low'
    END AS hdi_group,
    COUNT(*) AS n_countries
FROM hdi_data
GROUP BY hdi_group
ORDER BY hdi_group;
""", conn)

print(check.to_string(index=False))
print("Total countries:", check["n_countries"].sum())

  hdi_group  n_countries
1 Very High           59
     2 High           32
   3 Medium           23
      4 Low           16
Total countries: 130


**Conclusion (Q7):**

HDI is a number that shows how good life is in a country (how long people live,
how much schooling they get, how much income they have). A higher HDI means a more
developed country.

We split the 130 countries into four groups based on their HDI:
- Very High (most developed)
- High
- Medium
- Low (least developed)

Then we asked the database to show only the groups that have more than 20 countries.

The counts came out as:
- Very High: 59 countries
- High: 32 countries
- Medium: 23 countries
- Low: 16 countries

59, 32 and 23 are all more than 20, so they stayed. The Low group had only 16
countries, which is not more than 20, so it was removed.

In plain words: most countries in our data are in the higher (more developed)
groups, and the least developed group has the fewest countries.

This question also shows the difference between WHERE and HAVING:
- WHERE checks each country one by one, before the groups are made.
- HAVING checks the whole group, after the countries are counted.

We needed HAVING here because the rule ("more than 20 countries") is about the size
of the group, not about a single country.

### Q8. High corruption and low happiness

A low CPI score means high corruption (the scale runs 0 to 100, where higher is
cleaner). We join the corruption and happiness tables and look for countries that
are weak on both: CPI below 35 and happiness below 4.5. This shows where weak
governance and low well-being appear together.

In [12]:
query = """
SELECT
    c.country,
    c.cpi_score,
    hp.happiness_score
FROM corruption_data AS c
JOIN happiness_data AS hp
    ON c.code = hp.code
WHERE c.cpi_score < 35
  AND hp.happiness_score < 4.5
ORDER BY hp.happiness_score ASC;
"""

result = pd.read_sql(query, conn)
print(result.to_string(index=False))
print("Number of countries:", len(result))

     country  cpi_score  happiness_score
 Afghanistan         16            1.721
     Lebanon         23            2.707
Sierra Leone         34            3.245
      Malawi         34            3.421
     Comoros         20            3.566
  Bangladesh         24            3.886
       Egypt         30            3.977
        Togo         32            4.214
  Madagascar         25            4.228
        Mali         28            4.232
     Liberia         28            4.269
    Cambodia         20            4.341
      Uganda         25            4.372
       Kenya         30            4.470
        Chad         22            4.471
Number of countries: 15


**Conclusion (Q8):**

Quick reminder on the two scores:
- CPI score: 0 to 100. A lower score means more corruption. So a country with CPI
  below 35 is seen as fairly corrupt.
- Happiness score: roughly 2 to 8. A lower score means people are less happy.

We asked for countries that are weak on both at the same time: CPI below 35 and
happiness below 4.5. 15 countries matched.

At the very top is Afghanistan, with a CPI of 16 and a happiness score of 1.721,
the lowest in the whole dataset. It is followed by Lebanon, Sierra Leone, Malawi,
Comoros and others. Many of these are countries that have faced conflict, political
instability or serious economic trouble.

In plain words: countries where corruption is high also tend to be countries where
people report low happiness. The two problems often show up together.

One honest caution: this does not prove that corruption causes unhappiness. These
countries usually face many hard problems at once, such as poverty and conflict, so
we can only say the two move together here, not that one causes the other. We will
test relationships more carefully in the statistics stage.

### Q9. Create a combined view

So far the data lives in four separate tables, and every cross-table question needs
a JOIN. A view is a saved query with a name. We create one view that joins all four
tables on the country code, so later we can read everything from a single name
instead of writing the JOIN each time. A view does not copy the data, it is just a
saved shortcut.

In [16]:
conn.execute("DROP VIEW IF EXISTS country_overview;")

conn.execute("""
CREATE VIEW country_overview AS
SELECT
    h.code,
    h.country,
    h.hdi,
    h.life_expectancy,
    h.expected_schooling,
    h.mean_schooling,
    h.gni_per_capita,
    hp.happiness_score,
    hp.social_support_contrib,
    hp.freedom_contrib,
    hp.generosity_contrib,
    hp.corruption_contrib,
    e.gdp_per_capita,
    e.health_expenditure,
    e.unemployment,
    e.inflation,
    e.population,
    c.cpi_score
FROM hdi_data AS h
JOIN happiness_data AS hp ON h.code = hp.code
JOIN economic_data AS e   ON h.code = e.code
JOIN corruption_data AS c ON h.code = c.code;
""")

conn.commit()

pd.read_sql("SELECT country, hdi, gdp_per_capita, happiness_score, cpi_score FROM country_overview LIMIT 5;", conn)

,country,hdi,gdp_per_capita,happiness_score,cpi_score
0,Iceland,0.972,82138.789297,7.525,77
1,Norway,0.970,87497.217965,7.302,81
2,Switzerland,0.970,100623.549627,7.060,80
3,Denmark,0.962,68043.546697,7.583,89
4,Germany,0.959,54776.766824,6.719,77


**Conclusion (Q9):**

We created a view called country_overview that joins all four tables on the country
code. A view is just a saved query with a name, it does not copy any data.

Now, instead of writing a four-table JOIN every time, we can read everything from
this single name, just like reading from one table. The quick check shows the first
five countries with their HDI, GDP, happiness and corruption score all in one place.

This makes the next stages (charts and statistics) simpler, because all the data is
available through one clean source.